# Deng--Ginting Example 4: PoU Linear Head for the Hard-Curl PINN

This notebook validates a partition-of-unity (PoU) linear head on top of the frozen hard-curl PINN hidden features.  It does **not** modify the source hard-curl notebook or the IMPES simulator.  The pressure solve, CG face-flux extraction, Deng/NLR reconstruction, particular flux `q_p`, feature map, and metric conventions are reused from `impes_spe10_simulator.py` and the original hard-curl checkpoint.

Run order:

1. Setup and load the simulator/checkpoint.
2. Build the smooth PoU windows, reduced feature basis, and sparse face-flux design matrix.
3. Gate 0: warm-start identity.
4. Gate 1: no-drift fit at `S=0` with ridge sweep.
5. Gate 2: dual-CV conservation split.
6. Gate 3: drift tests at `t=0.05` and `t=0.10` using exp5 NLR snapshots.
7. Optional ablation.
8. Save the PoU checkpoint and metric JSON files.


In [ ]:
import json
import math
import os
import pathlib
import sys
import time
from dataclasses import asdict

import numpy as np
import scipy.sparse as sp
from scipy.io import loadmat
from scipy.sparse.linalg import splu
import torch

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-cache")

# Make the notebook robust to being run either from the project root or from this folder.
NOTEBOOK_DIR = pathlib.Path.cwd()
if not (NOTEBOOK_DIR / "impes_spe10_simulator.py").exists():
    NOTEBOOK_DIR = pathlib.Path("fenicsx/code/fracture problem").resolve()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import impes_spe10_simulator as impes

CASE_DIR = NOTEBOOK_DIR / "case3_ecmor"
OUT_DIR = NOTEBOOK_DIR / "result_pou_head_spe10_Q1"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_CHECKPOINT = CASE_DIR / "hardcurl_pinn_spe10_Q1_64x64.pt"
POU_CHECKPOINT = CASE_DIR / "hardcurl_pinn_spe10_Q1_64x64_pou.pt"
GATE_METRICS_JSON = CASE_DIR / "hardcurl_pinn_spe10_Q1_64x64_pou_metrics.json"

# Default PoU configuration requested in the spec.
POU_WINDOW_SHAPE = (16, 16)
POU_OVERLAP = 0.50
POU_R = 16
RIDGE_SWEEP_REL = [1.0e-8, 1.0e-6, 1.0e-4]
DEFAULT_RIDGE_REL = 1.0e-8

# Full L-BFGS is a reference ceiling, not the cheap path.
RUN_FULL_LBFGS_REFERENCE = False
FULL_LBFGS_MAX_ITER = 300
FULL_LBFGS_PRINT_EVERY = 25

RUN_ABLATION = False
ABLATION_WINDOW_SHAPES = [(4, 4), (8, 8), (16, 16)]
ABLATION_R_VALUES = [8, 16, 32, 96]

np.set_printoptions(precision=6, suppress=False)
torch.set_default_dtype(torch.float64)
print("Notebook dir:", NOTEBOOK_DIR)
print("Checkpoint:", SOURCE_CHECKPOINT)


Notebook dir: /home/muchamad/PhD/fenicsx/code/fracture problem
Checkpoint: /home/muchamad/PhD/fenicsx/code/fracture problem/case3_ecmor/hardcurl_pinn_spe10_Q1_64x64.pt


In [2]:
# Instantiate one simulator that exposes the Q1 pressure solve, dual faces, CG/Deng fluxes,
# and the frozen hard-curl flux model.  validate_deng=True precomputes the local Deng geometry.
cfg = impes.ImpesConfig(
    method="PINN",
    pinn_mode="frozen",
    validate_deng=True,
    t0_checkpoint_path=str(SOURCE_CHECKPOINT),
    out_dir=str(OUT_DIR / "setup_sim"),
    save_every=0,
    print_every=0,
)
sim = impes.ImpesSpe10Simulator(cfg)
flux_model = sim.flux_model
assert flux_model is not None

n_faces = len(sim.dual_owner)
print(f"Dual faces: {n_faces}, dual CVs: {sim.n_nodes}, cells: {sim.ncell}")
print(f"Hidden dim: {flux_model.hidden_dim}, dtype: {flux_model.dtype}")


Loaded hard-curl PINN checkpoint: /home/muchamad/PhD/fenicsx/code/fracture problem/case3_ecmor/hardcurl_pinn_spe10_Q1_64x64.pt
Dual faces: 8580, dual CVs: 4225, cells: 4096
Hidden dim: 96, dtype: torch.float64


In [3]:
# MRST1024 / CVFEM576 coarse-dual oracle loaders, ported from the source notebook.
def _axis_segment_lookup_for_raw_faces(target_p1, target_p2, n_raw, tol=1.0e-10):
    target_p1 = np.asarray(target_p1, dtype=float).reshape(-1, 2)
    target_p2 = np.asarray(target_p2, dtype=float).reshape(-1, 2)
    lookup = {}
    duplicate_keys = 0
    skipped_targets = 0
    for j, (a, b) in enumerate(zip(target_p1, target_p2)):
        if abs(a[0] - b[0]) <= tol and abs(a[1] - b[1]) > tol:
            axis = "V"
            line = int(round(0.5 * (a[0] + b[0]) * n_raw))
            lo = int(round(min(a[1], b[1]) * n_raw))
            hi = int(round(max(a[1], b[1]) * n_raw))
        elif abs(a[1] - b[1]) <= tol and abs(a[0] - b[0]) > tol:
            axis = "H"
            line = int(round(0.5 * (a[1] + b[1]) * n_raw))
            lo = int(round(min(a[0], b[0]) * n_raw))
            hi = int(round(max(a[0], b[0]) * n_raw))
        else:
            skipped_targets += 1
            continue
        if hi <= lo:
            skipped_targets += 1
            continue
        lo = max(0, lo)
        hi = min(int(n_raw), hi)
        for m in range(lo, hi):
            key = (axis, int(line), int(m))
            if key in lookup and lookup[key] != j:
                duplicate_keys += 1
            lookup[key] = j
    return lookup, {"duplicate_keys": int(duplicate_keys), "skipped_targets": int(skipped_targets)}


def sum_mrst1024_primal_flux_to_coarse_dual_faces(path, target_p1, target_p2, target_normal):
    path = pathlib.Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing MRST-1024 oracle flux file: {path}")
    data = loadmat(path, squeeze_me=True, struct_as_record=False)
    required = ["face_flux_1024", "face_centroid_1024", "face_normal_1024"]
    missing = [name for name in required if name not in data]
    if missing:
        raise KeyError(f"{path.name} is missing required raw-1024 fields: {missing}")
    if "meta_pressure_celldim" in data:
        dims = np.asarray(data["meta_pressure_celldim"], dtype=np.int64).reshape(-1)
        n_raw = int(dims[0])
    else:
        n_faces_raw = int(np.asarray(data["face_flux_1024"]).size)
        n_raw = int(round((np.sqrt(1.0 + 2.0 * n_faces_raw) - 1.0) / 2.0))
        if 2 * n_raw * (n_raw + 1) != n_faces_raw:
            raise ValueError(f"Could not infer raw Cartesian grid size from {n_faces_raw} faces")

    target_normal = np.asarray(target_normal, dtype=float).reshape(-1, 2)
    lookup, lookup_diag = _axis_segment_lookup_for_raw_faces(target_p1, target_p2, n_raw)
    face_flux = np.asarray(data["face_flux_1024"], dtype=float).reshape(-1)
    face_centroid = np.asarray(data["face_centroid_1024"], dtype=float).reshape(-1, 2)
    face_normal = np.asarray(data["face_normal_1024"], dtype=float).reshape(-1, 2)
    Fout = np.zeros(len(target_normal), dtype=np.float64)
    hit_count = np.zeros(len(target_normal), dtype=np.int64)
    used_raw_faces = 0
    skipped_not_on_target = 0
    skipped_orientation = 0
    for F, ctr, nrm in zip(face_flux, face_centroid, face_normal):
        if abs(nrm[0]) >= 0.5:
            axis = "V"
            line = int(round(float(ctr[0]) * n_raw))
            m = int(np.floor(np.clip(float(ctr[1]) * n_raw, 0.0, n_raw - 1.0e-12)))
        elif abs(nrm[1]) >= 0.5:
            axis = "H"
            line = int(round(float(ctr[1]) * n_raw))
            m = int(np.floor(np.clip(float(ctr[0]) * n_raw, 0.0, n_raw - 1.0e-12)))
        else:
            skipped_orientation += 1
            continue
        j = lookup.get((axis, int(line), int(m)))
        if j is None:
            skipped_not_on_target += 1
            continue
        orient = float(np.dot(nrm, target_normal[j]))
        if abs(orient) < 0.5:
            skipped_orientation += 1
            continue
        Fout[j] += float(F) * (1.0 if orient > 0.0 else -1.0)
        hit_count[j] += 1
        used_raw_faces += 1
    diagnostics = {
        "file": str(path),
        "kind": "MRST1024 raw primal-face sum",
        "n_raw": int(n_raw),
        "raw_faces_total": int(face_flux.size),
        "raw_faces_used": int(used_raw_faces),
        "target_faces": int(len(target_normal)),
        "matched_target_faces": int(np.count_nonzero(hit_count)),
        "min_raw_faces_per_matched_target": int(np.min(hit_count[hit_count > 0])) if np.any(hit_count > 0) else 0,
        "max_raw_faces_per_matched_target": int(np.max(hit_count)) if hit_count.size else 0,
        "skipped_not_on_target": int(skipped_not_on_target),
        "skipped_orientation": int(skipped_orientation),
        **lookup_diag,
    }
    return Fout, diagnostics


def load_oracle_fluxes():
    oracle = {}
    diagnostics = {}
    mrst1024_path = CASE_DIR / "case3_mrst_export_spe10_ref_cvfem.mat"
    if mrst1024_path.exists():
        F, diag = sum_mrst1024_primal_flux_to_coarse_dual_faces(
            mrst1024_path, sim.dual_p1, sim.dual_p2, sim.dual_normal
        )
        oracle["MRST1024_flux"] = F.astype(np.float64)
        diagnostics["MRST1024_flux"] = diag
    cvfem_path = CASE_DIR / "case3_cvfem_oracle_spe10_576_dual64.mat"
    if cvfem_path.exists():
        data = loadmat(cvfem_path, squeeze_me=True, struct_as_record=False)
        if "face_flux" in data:
            F = np.asarray(data["face_flux"], dtype=np.float64).reshape(-1)
            if F.size == n_faces:
                oracle["CVFEM576_flux"] = F
                diagnostics["CVFEM576_flux"] = {
                    "file": str(cvfem_path),
                    "kind": "CVFEM576 direct coarse-dual face_flux",
                    "target_faces": int(n_faces),
                    "matched_target_faces": int(n_faces),
                }
    if not oracle:
        raise FileNotFoundError("No MRST1024/CVFEM576 oracle flux data found in case3_ecmor")
    return oracle, diagnostics

oracle_flux_by_name, oracle_flux_diagnostics = load_oracle_fluxes()
ORACLE_NAME = "MRST1024_flux" if "MRST1024_flux" in oracle_flux_by_name else sorted(oracle_flux_by_name)[0]
F_oracle = oracle_flux_by_name[ORACLE_NAME]
print("Loaded oracle fluxes:", sorted(oracle_flux_by_name))
print("Using primary oracle:", ORACLE_NAME)
print(json.dumps(oracle_flux_diagnostics[ORACLE_NAME], indent=2))


Loaded oracle fluxes: ['CVFEM576_flux', 'MRST1024_flux']
Using primary oracle: MRST1024_flux
{
  "file": "/home/muchamad/PhD/fenicsx/code/fracture problem/case3_ecmor/case3_mrst_export_spe10_ref_cvfem.mat",
  "kind": "MRST1024 raw primal-face sum",
  "n_raw": 1024,
  "raw_faces_total": 2099200,
  "raw_faces_used": 135168,
  "target_faces": 8580,
  "matched_target_faces": 8580,
  "min_raw_faces_per_matched_target": 8,
  "max_raw_faces_per_matched_target": 16,
  "skipped_not_on_target": 1964032,
  "skipped_orientation": 0,
  "duplicate_keys": 0,
  "skipped_targets": 0
}


In [4]:
def face_stats(err):
    err = np.asarray(err, dtype=np.float64).reshape(-1)
    finite = np.isfinite(err)
    err = err[finite]
    return {
        "n": int(err.size),
        "l2": float(np.sqrt(np.sum(err * err))),
        "rmse": float(np.sqrt(np.mean(err * err))) if err.size else float("nan"),
        "mean_abs": float(np.mean(np.abs(err))) if err.size else float("nan"),
        "median_abs": float(np.median(np.abs(err))) if err.size else float("nan"),
        "max_abs": float(np.max(np.abs(err))) if err.size else float("nan"),
    }


def residual_split_stats(F):
    r = sim.dual_residual(np.asarray(F, dtype=np.float64))
    masks = {
        "all": np.ones_like(r, dtype=bool),
        "interior": sim.dual_interior_mask,
        "source": sim.dual_source_mask,
        "boundary": sim.dual_boundary_mask,
    }
    out = {}
    for name, mask in masks.items():
        arr = r[mask]
        out[name] = face_stats(arr)
        out[name]["sum"] = float(np.sum(arr))
    return out


def print_metric_table(rows, title):
    print("\n" + title)
    print(f"{'name':<30} {'L2':>13} {'RMSE':>13} {'max|e|':>13} {'mean|e|':>13}")
    for row in rows:
        print(
            f"{row['name']:<30} {row['l2']:13.6e} {row['rmse']:13.6e} "
            f"{row['max_abs']:13.6e} {row['mean_abs']:13.6e}"
        )


def json_ready(obj):
    if isinstance(obj, dict):
        return {str(k): json_ready(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [json_ready(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    return obj


In [5]:
# Build hidden features at every dual-face endpoint and construct the reduced basis P.
with torch.no_grad():
    H_a = flux_model.model.hidden_features(flux_model.feat_a).detach().cpu().numpy().astype(np.float64)
    H_b = flux_model.model.hidden_features(flux_model.feat_b).detach().cpu().numpy().astype(np.float64)
final_layer = list(flux_model.model.net.children())[-1]
w_last = final_layer.weight.detach().cpu().numpy().reshape(-1).astype(np.float64)
if np.linalg.norm(w_last) == 0.0:
    raise RuntimeError("Checkpoint final-layer weight is zero; PoU identity basis cannot be built.")


def build_projection_matrix(H_a, H_b, w, r):
    H = np.vstack([H_a, H_b]).astype(np.float64)
    w = np.asarray(w, dtype=np.float64).reshape(-1)
    hidden_dim = w.size
    r = int(r)
    if r < 1 or r > hidden_dim:
        raise ValueError(f"r must be in [1,{hidden_dim}], got {r}")
    ww = float(np.dot(w, w))
    Hc = H - np.mean(H, axis=0, keepdims=True)
    Hc = Hc - np.outer(Hc @ w, w) / ww
    _, _, Vt = np.linalg.svd(Hc, full_matrices=False)
    modes = []
    for cand in Vt:
        v = np.asarray(cand, dtype=np.float64).copy()
        v -= w * (float(np.dot(v, w)) / ww)
        for u in modes:
            v -= u * float(np.dot(v, u))
        nv = float(np.linalg.norm(v))
        if nv > 1.0e-12:
            modes.append(v / nv)
        if len(modes) >= r - 1:
            break
    if len(modes) < r - 1:
        # Robust fallback: complete the basis from coordinate vectors, orthogonal to w and prior modes.
        for j in range(hidden_dim):
            v = np.zeros(hidden_dim, dtype=np.float64)
            v[j] = 1.0
            v -= w * (float(np.dot(v, w)) / ww)
            for u in modes:
                v -= u * float(np.dot(v, u))
            nv = float(np.linalg.norm(v))
            if nv > 1.0e-12:
                modes.append(v / nv)
            if len(modes) >= r - 1:
                break
    if len(modes) != r - 1:
        raise RuntimeError(f"Could only build {1 + len(modes)} basis vectors for r={r}")
    return np.column_stack([w] + modes).astype(np.float64)

P_default = build_projection_matrix(H_a, H_b, w_last, POU_R)
print("P shape:", P_default.shape)
print("First basis vector identity error ||P[:,0]-w_last||:", np.linalg.norm(P_default[:, 0] - w_last))


P shape: (96, 16)
First basis vector identity error ||P[:,0]-w_last||: 0.0


In [6]:
class PoULinearHead:
    """Sparse PoU linear head for endpoint-difference hard-curl face fluxes."""

    def __init__(self, flux_model, H_a, H_b, P, window_shape=(8, 8), overlap=0.5):
        self.flux_model = flux_model
        self.H_a = np.asarray(H_a, dtype=np.float64)
        self.H_b = np.asarray(H_b, dtype=np.float64)
        self.P = np.asarray(P, dtype=np.float64)
        self.r = int(self.P.shape[1])
        self.window_shape = tuple(int(v) for v in window_shape)
        self.overlap = float(overlap)
        self.p1 = np.asarray(flux_model.p1, dtype=np.float64)
        self.p2 = np.asarray(flux_model.p2, dtype=np.float64)
        self.sign = flux_model.sign.detach().cpu().numpy().astype(np.float64).reshape(-1)
        self.qp = np.asarray(flux_model.qp_flux_np, dtype=np.float64).reshape(-1)
        self.K = int(self.window_shape[0] * self.window_shape[1])
        self.theta_bar = np.zeros((self.K, self.r), dtype=np.float64)
        self.theta_bar[:, 0] = 1.0
        self.theta = self.theta_bar.copy()
        self._factor_cache = {}
        self._build_design()

    @staticmethod
    def _cosine_bump_1d(x, centers, radius):
        x = np.asarray(x, dtype=np.float64).reshape(-1, 1)
        centers = np.asarray(centers, dtype=np.float64).reshape(1, -1)
        z = np.abs(x - centers) / float(radius)
        out = np.zeros_like(z, dtype=np.float64)
        mask = z < 1.0
        out[mask] = 0.5 * (1.0 + np.cos(np.pi * z[mask]))
        return out

    def window_weights(self, points):
        pts = np.asarray(points, dtype=np.float64).reshape(-1, 2)
        nxw, nyw = self.window_shape
        cx = np.linspace(0.0, 1.0, nxw) if nxw > 1 else np.array([0.5])
        cy = np.linspace(0.0, 1.0, nyw) if nyw > 1 else np.array([0.5])
        # Support radius = grid spacing gives smooth 50%-overlap cosine bumps.
        rx = (1.0 / max(nxw - 1, 1)) / max(1.0 - self.overlap, 1.0e-12) * 0.5
        ry = (1.0 / max(nyw - 1, 1)) / max(1.0 - self.overlap, 1.0e-12) * 0.5
        if nxw == 1:
            wx = np.ones((len(pts), 1), dtype=np.float64)
        else:
            wx = self._cosine_bump_1d(pts[:, 0], cx, rx)
        if nyw == 1:
            wy = np.ones((len(pts), 1), dtype=np.float64)
        else:
            wy = self._cosine_bump_1d(pts[:, 1], cy, ry)
        W = (wy[:, :, None] * wx[:, None, :]).reshape(len(pts), nyw * nxw)
        denom = np.sum(W, axis=1, keepdims=True)
        if np.any(denom <= 0.0):
            bad = np.flatnonzero(denom[:, 0] <= 0.0)[:10]
            raise RuntimeError(f"PoU windows do not cover endpoint(s): {bad}")
        return W / denom

    def _build_design(self):
        t0 = time.perf_counter()
        Wa = self.window_weights(self.p1)
        Wb = self.window_weights(self.p2)
        self.window_sum_error = max(
            float(np.max(np.abs(np.sum(Wa, axis=1) - 1.0))),
            float(np.max(np.abs(np.sum(Wb, axis=1) - 1.0))),
        )
        phi_a = self.H_a @ self.P
        phi_b = self.H_b @ self.P
        rows = []
        cols = []
        vals = []
        n = self.p1.shape[0]
        ar = np.arange(self.r, dtype=np.int64)
        for f in range(n):
            for W, phi, sgn in ((Wb, phi_b, self.sign[f]), (Wa, phi_a, -self.sign[f])):
                active = np.flatnonzero(W[f] > 1.0e-14)
                if active.size == 0:
                    continue
                block_cols = active[:, None] * self.r + ar[None, :]
                block_vals = sgn * W[f, active][:, None] * phi[f][None, :]
                rows.append(np.full(block_cols.size, f, dtype=np.int64))
                cols.append(block_cols.reshape(-1).astype(np.int64))
                vals.append(block_vals.reshape(-1).astype(np.float64))
        self.Phi = sp.coo_matrix(
            (np.concatenate(vals), (np.concatenate(rows), np.concatenate(cols))),
            shape=(n, self.K * self.r),
            dtype=np.float64,
        ).tocsc()
        self.build_s = time.perf_counter() - t0
        self.theta_bar_vec = self.theta_bar.reshape(-1)
        self.Phi_theta_bar = np.asarray(self.Phi @ self.theta_bar_vec).reshape(-1)

    def prediction(self, theta=None):
        if theta is None:
            theta = self.theta
        theta_vec = np.asarray(theta, dtype=np.float64).reshape(-1)
        return self.qp + np.asarray(self.Phi @ theta_vec).reshape(-1)

    def factorize(self, ridge_rel):
        ridge_rel = float(ridge_rel)
        if ridge_rel in self._factor_cache:
            return self._factor_cache[ridge_rel]
        t0 = time.perf_counter()
        normal = (self.Phi.T @ self.Phi).tocsc()
        diag_max = float(np.max(np.abs(normal.diagonal())))
        ridge_abs = ridge_rel * max(diag_max, 1.0)
        A = normal + ridge_abs * sp.eye(normal.shape[0], dtype=np.float64, format="csc")
        lu = splu(A)
        out = {"lu": lu, "ridge_abs": ridge_abs, "diag_max": diag_max, "factor_s": time.perf_counter() - t0}
        self._factor_cache[ridge_rel] = out
        return out

    def fit(self, target, ridge_rel=1.0e-6, anchor=None):
        target = np.asarray(target, dtype=np.float64).reshape(-1)
        anchor_vec = self.theta_bar_vec if anchor is None else np.asarray(anchor, dtype=np.float64).reshape(-1)
        fact = self.factorize(ridge_rel)
        t0 = time.perf_counter()
        y = target - self.qp
        rhs = np.asarray(self.Phi.T @ y).reshape(-1) + fact["ridge_abs"] * anchor_vec
        theta_vec = fact["lu"].solve(rhs)
        pred = self.qp + np.asarray(self.Phi @ theta_vec).reshape(-1)
        fit_s = time.perf_counter() - t0
        self.theta = theta_vec.reshape(self.K, self.r)
        return {
            "theta": self.theta.copy(),
            "prediction": pred,
            "ridge_rel": float(ridge_rel),
            "ridge_abs": float(fact["ridge_abs"]),
            "factor_s": float(fact["factor_s"]),
            "fit_s": float(fit_s),
            "dof": int(self.K * self.r),
        }


def make_pou_head(window_shape=POU_WINDOW_SHAPE, r=POU_R, overlap=POU_OVERLAP):
    P = build_projection_matrix(H_a, H_b, w_last, int(r))
    return PoULinearHead(flux_model, H_a, H_b, P, window_shape=window_shape, overlap=overlap)

pou = make_pou_head(POU_WINDOW_SHAPE, POU_R, POU_OVERLAP)
print(f"PoU design: Phi={pou.Phi.shape}, nnz={pou.Phi.nnz}, build_s={pou.build_s:.3f}, window_sum_error={pou.window_sum_error:.3e}")


PoU design: Phi=(8580, 1024), nnz=565760, build_s=0.330, window_sum_error=2.220e-16


In [7]:
# Gate 0 -- warm-start identity.
F_checkpoint = flux_model.prediction_numpy().astype(np.float64)
F_warm = pou.prediction(pou.theta_bar)
gate0 = face_stats(F_warm - F_checkpoint)
print_metric_table([{**gate0, "name": "PoU warm-start - checkpoint"}], "Gate 0: warm-start identity")
print("max warm-start identity error:", gate0["max_abs"])
if gate0["max_abs"] > 1.0e-12:
    raise AssertionError("Gate 0 failed: PoU warm-start does not reproduce checkpoint face fluxes to <= 1e-12")



Gate 0: warm-start identity
name                                      L2          RMSE        max|e|       mean|e|
PoU warm-start - checkpoint     1.057709e-13  1.141885e-15  8.326673e-15  7.358103e-16
max warm-start identity error: 8.326672684688674e-15


In [8]:
# Gate 1 -- t=0 fit to CG dual-face fluxes and compare with the primary oracle.
coeff0 = sim.kappa_base.copy().astype(np.float64)
p0 = sim.solve_pressure(coeff0)
F_cg0 = sim.face_flux_cg(p0, coeff0).astype(np.float64)
F_nlr0 = sim.face_flux_deng(p0, coeff0).astype(np.float64)
F_orig0 = F_checkpoint.copy()

base_rows = []
for name, F_ref, F_cmp in [
    ("CG vs oracle", F_oracle, F_cg0),
    ("original PINN vs CG", F_cg0, F_orig0),
    ("original PINN vs oracle", F_oracle, F_orig0),
    ("NLR vs CG", F_cg0, F_nlr0),
    ("NLR vs oracle", F_oracle, F_nlr0),
]:
    base_rows.append({"name": name, **face_stats(F_cmp - F_ref)})

ridge_results = []
for ridge_rel in RIDGE_SWEEP_REL:
    fit = pou.fit(F_cg0, ridge_rel=ridge_rel)
    F_pou = fit["prediction"]
    row_cg = {"name": f"PoU r={POU_R} ridge={ridge_rel:g} vs CG", **face_stats(F_pou - F_cg0)}
    row_oracle = {"name": f"PoU r={POU_R} ridge={ridge_rel:g} vs oracle", **face_stats(F_pou - F_oracle)}
    ridge_results.append({
        "ridge_rel": float(ridge_rel),
        "ridge_abs": float(fit["ridge_abs"]),
        "factor_s": float(fit["factor_s"]),
        "fit_s": float(fit["fit_s"]),
        "dof": int(fit["dof"]),
        "vs_cg": row_cg,
        "vs_oracle": row_oracle,
        "theta": fit["theta"].copy(),
        "prediction": F_pou.copy(),
    })

orig_oracle_rmse = face_stats(F_orig0 - F_oracle)["rmse"]
acceptable = [r for r in ridge_results if r["vs_oracle"]["rmse"] <= 1.05 * orig_oracle_rmse]
if acceptable:
    selected = min(acceptable, key=lambda r: r["vs_cg"]["rmse"])
else:
    selected = min(ridge_results, key=lambda r: r["vs_oracle"]["rmse"])

pou.theta = selected["theta"].copy()
POU_SELECTED_RIDGE_REL = float(selected["ridge_rel"])
F_pou_gate1 = selected["prediction"].copy()

print_metric_table(base_rows, "Gate 1 base metrics")
print_metric_table(
    [item for r in ridge_results for item in (r["vs_cg"], r["vs_oracle"])],
    "Gate 1 PoU ridge sweep"
)
print(f"Selected ridge_rel={POU_SELECTED_RIDGE_REL:g} (oracle RMSE original={orig_oracle_rmse:.6e})")
if selected["vs_cg"]["rmse"] < face_stats(F_orig0 - F_cg0)["rmse"] and selected["vs_oracle"]["rmse"] > orig_oracle_rmse:
    print("Overfitting guard: PoU improves vs CG but degrades vs oracle relative to original PINN; prefer larger ridge/report trend.")



Gate 1 base metrics
name                                      L2          RMSE        max|e|       mean|e|
CG vs oracle                    2.622360e-01  2.831057e-03  6.316408e-02  7.281552e-04
original PINN vs CG             2.339839e-01  2.526053e-03  4.426141e-02  1.557473e-03
original PINN vs oracle         2.490371e-01  2.688564e-03  4.054884e-02  1.609240e-03
NLR vs CG                       1.827810e-01  1.973274e-03  5.155973e-02  4.445648e-04
NLR vs oracle                   1.899945e-01  2.051150e-03  3.731428e-02  5.801869e-04

Gate 1 PoU ridge sweep
name                                      L2          RMSE        max|e|       mean|e|
PoU r=16 ridge=1e-08 vs CG      2.241794e-01  2.420204e-03  4.413607e-02  1.425630e-03
PoU r=16 ridge=1e-08 vs oracle  2.384185e-01  2.573927e-03  4.031307e-02  1.467652e-03
PoU r=16 ridge=1e-06 vs CG      2.255440e-01  2.434936e-03  4.413403e-02  1.446265e-03
PoU r=16 ridge=1e-06 vs oracle  2.397181e-01  2.587958e-03  4.039659e-02  1.489113e-0

In [9]:
# Gate 2 -- conservation of the Gate-1 PoU flux on the Deng dual CVs.
gate2_stats = residual_split_stats(F_pou_gate1)
for part, st in gate2_stats.items():
    print(f"{part:<9} RMSE={st['rmse']:.6e} max={st['max_abs']:.6e} sum={st['sum']:+.6e}")
max_gate2 = max(st["max_abs"] for st in gate2_stats.values())
if max_gate2 > 1.0e-13:
    raise AssertionError(f"Gate 2 failed: max split residual is {max_gate2:.3e} > 1e-13")


all       RMSE=1.944202e-15 max=1.205871e-14 sum=-3.775571e-15
interior  RMSE=1.936754e-15 max=1.205871e-14 sum=-4.112596e-15
source    RMSE=2.695687e-15 max=5.440093e-15 sum=-2.553513e-15
boundary  RMSE=2.056225e-15 max=1.130595e-14 sum=+3.370242e-16


In [10]:
def make_flux_model(pinn_mode="last_layer", lbfgs_max_iter=300):
    cfg = impes.ImpesConfig(
        method="PINN",
        pinn_mode=pinn_mode,
        validate_deng=True,
        t0_checkpoint_path=str(SOURCE_CHECKPOINT),
        out_dir=str(OUT_DIR / f"tmp_{pinn_mode}_{time.time_ns()}"),
        save_every=0,
        print_every=0,
        lbfgs_max_iter=int(lbfgs_max_iter),
        lbfgs_print_every=FULL_LBFGS_PRINT_EVERY,
        lbfgs_early_stop_patience=75,
        lbfgs_rel_min_delta=1.0e-4,
    )
    s = impes.ImpesSpe10Simulator(cfg)
    return s.flux_model


def drift_target_from_snapshot(snapshot_path):
    S = np.load(snapshot_path).reshape(-1).astype(np.float64)
    S_cell = sim.project_dual_to_cells(S)
    coeff = sim.kappa_base * impes.mobility_factor(S_cell, sim.cfg.M)
    p = sim.solve_pressure(coeff)
    F_cg = sim.face_flux_cg(p, coeff).astype(np.float64)
    F_nlr = sim.face_flux_deng(p, coeff).astype(np.float64)
    return S, coeff, p, F_cg, F_nlr


def run_gate3_for_snapshot(label, snapshot_path, pou_head, ridge_rel):
    print(f"\nGate 3 drift snapshot {label}: {snapshot_path}")
    S, coeff, p, F_target, F_nlr = drift_target_from_snapshot(snapshot_path)
    rows = []
    cons = {}

    # (a) frozen checkpoint baseline
    F_frozen = F_checkpoint.copy()
    rows.append({"name": "frozen checkpoint vs CG", **face_stats(F_frozen - F_target)})
    cons["frozen checkpoint"] = residual_split_stats(F_frozen)

    # Diagnostic: Gate-1 PoU state without a drift update.
    F_pou_frozen = pou_head.prediction(pou_head.theta)
    rows.append({"name": "PoU Gate-1 state vs CG", **face_stats(F_pou_frozen - F_target)})
    cons["PoU Gate-1 state"] = residual_split_stats(F_pou_frozen)

    # (b) global linear-last direct solve from the simulator implementation.
    linear_model = make_flux_model("last_layer")
    t0 = time.perf_counter()
    F_linear = linear_model.update_last_layer(F_target)
    linear_s = time.perf_counter() - t0
    rows.append({"name": "global linear-last vs CG", **face_stats(F_linear - F_target), "fit_s": linear_s})
    cons["global linear-last"] = residual_split_stats(F_linear)

    # (c) PoU LSQ update.
    fit = pou_head.fit(F_target, ridge_rel=ridge_rel)
    F_pou = fit["prediction"]
    rows.append({"name": "PoU LSQ vs CG", **face_stats(F_pou - F_target), "fit_s": fit["fit_s"], "factor_s": fit["factor_s"]})
    cons["PoU LSQ"] = residual_split_stats(F_pou)

    # (d) full L-BFGS reference ceiling, capped.
    if RUN_FULL_LBFGS_REFERENCE:
        full_model = make_flux_model("full", lbfgs_max_iter=FULL_LBFGS_MAX_ITER)
        t0 = time.perf_counter()
        F_full = full_model.update_full(F_target)
        full_s = time.perf_counter() - t0
        rep = full_model.report()
        rows.append({
            "name": "full L-BFGS vs CG",
            **face_stats(F_full - F_target),
            "fit_s": full_s,
            "iterations": rep.get("pinn_iterations"),
            "stop": rep.get("pinn_stop_reason"),
        })
        cons["full L-BFGS"] = residual_split_stats(F_full)

    # (e) NLR/Deng local reconstruction.
    rows.append({"name": "NLR vs CG", **face_stats(F_nlr - F_target)})
    cons["NLR"] = residual_split_stats(F_nlr)

    print_metric_table(rows, f"Gate 3 {label}: face RMSE vs drifted CG target")
    print("Conservation split after drift fits:")
    for name, split in cons.items():
        print(
            f"  {name:<22} int={split['interior']['rmse']:.3e} "
            f"src={split['source']['rmse']:.3e} bnd={split['boundary']['rmse']:.3e} all={split['all']['rmse']:.3e}"
        )
    return {"label": label, "snapshot": str(snapshot_path), "rows": rows, "conservation": cons}

snapshots = {
    "t=0.05": NOTEBOOK_DIR / "impes_runs/exp5_NLR_M1/S_step5000.npy",
    "t=0.10": NOTEBOOK_DIR / "impes_runs/exp5_NLR_M1/S_step10000.npy",
}
missing = [str(p) for p in snapshots.values() if not pathlib.Path(p).exists()]
if missing:
    raise FileNotFoundError("Gate 3 needs exp5 NLR snapshots; missing: " + ", ".join(missing))

gate3_results = []
# Reset to Gate-1 selected state before drift sequence.
pou.theta = selected["theta"].copy()
for label, path in snapshots.items():
    gate3_results.append(run_gate3_for_snapshot(label, path, pou, POU_SELECTED_RIDGE_REL))



Gate 3 drift snapshot t=0.05: /home/muchamad/PhD/fenicsx/code/fracture problem/impes_runs/exp5_NLR_M1/S_step5000.npy
Loaded hard-curl PINN checkpoint: /home/muchamad/PhD/fenicsx/code/fracture problem/case3_ecmor/hardcurl_pinn_spe10_Q1_64x64.pt
Loaded hard-curl PINN checkpoint: /home/muchamad/PhD/fenicsx/code/fracture problem/case3_ecmor/hardcurl_pinn_spe10_Q1_64x64.pt
  L-BFGS call   300: face-rmse=2.452111e-03, best=2.452111e-03, elapsed=24.6s

Gate 3 t=0.05: face RMSE vs drifted CG target
name                                      L2          RMSE        max|e|       mean|e|
frozen checkpoint vs CG         2.915417e-01  3.147437e-03  4.511734e-02  1.934192e-03
PoU Gate-1 state vs CG          2.839895e-01  3.065905e-03  4.499200e-02  1.811688e-03
global linear-last vs CG        2.814175e-01  3.038137e-03  4.202439e-02  1.910202e-03
PoU LSQ vs CG                   2.335166e-01  2.521007e-03  4.447483e-02  1.521049e-03
full L-BFGS vs CG               2.262620e-01  2.442687e-03  4.398315

In [11]:
# Ablation -- run only after the gates above pass.
def run_pou_ablation():
    records = []
    drift_targets = {}
    for label, path in snapshots.items():
        _, _, _, F_target, _ = drift_target_from_snapshot(path)
        drift_targets[label] = F_target
    for window_shape in ABLATION_WINDOW_SHAPES:
        for r in ABLATION_R_VALUES:
            t0 = time.perf_counter()
            head = make_pou_head(window_shape=window_shape, r=r, overlap=POU_OVERLAP)
            build_s = time.perf_counter() - t0
            # Use the selected ridge policy when possible; otherwise default.
            ridge_rel = POU_SELECTED_RIDGE_REL if "POU_SELECTED_RIDGE_REL" in globals() else DEFAULT_RIDGE_REL
            # Gate-0 identity for this ablation config.
            warm = face_stats(head.prediction(head.theta_bar) - F_checkpoint)
            if warm["max_abs"] > 1.0e-12:
                raise AssertionError(f"Ablation Gate 0 failed for windows={window_shape}, r={r}: {warm['max_abs']:.3e}")
            # Factor once using first target; subsequent fits reuse it.
            row = {
                "window_shape": tuple(window_shape),
                "r": int(r),
                "dof": int(np.prod(window_shape) * r),
                "build_s": float(build_s),
                "warm_identity_max": warm["max_abs"],
            }
            fit_times = []
            for label, F_target in drift_targets.items():
                fit = head.fit(F_target, ridge_rel=ridge_rel)
                err = face_stats(fit["prediction"] - F_target)
                row[f"{label}_rmse"] = err["rmse"]
                row[f"{label}_l2"] = err["l2"]
                row[f"{label}_max_abs"] = err["max_abs"]
                row[f"{label}_fit_s"] = fit["fit_s"]
                fit_times.append(fit["fit_s"])
                row["factor_s"] = fit["factor_s"]
                split = residual_split_stats(fit["prediction"])
                row[f"{label}_Rxi_all_rmse"] = split["all"]["rmse"]
            row["mean_fit_s"] = float(np.mean(fit_times))
            records.append(row)
            print(
                f"ablation windows={window_shape}, r={r:2d}, dof={row['dof']:5d}, "
                f"rmse05={row['t=0.05_rmse']:.6e}, rmse10={row['t=0.10_rmse']:.6e}, "
                f"fit_s={row['mean_fit_s']:.4e}, factor_s={row['factor_s']:.3f}"
            )
    return records

ablation_results = run_pou_ablation() if RUN_ABLATION else []
if ablation_results:
    # Recommend by mean drift RMSE, with fit time as a secondary key.
    def score(rec):
        return (rec["t=0.05_rmse"] + rec["t=0.10_rmse"], rec["mean_fit_s"])
    recommended = min(ablation_results, key=score)
    print("\nRecommended config:", recommended)
else:
    recommended = {"window_shape": POU_WINDOW_SHAPE, "r": POU_R, "ridge_rel": POU_SELECTED_RIDGE_REL}


ablation windows=(4, 4), r= 8, dof=  128, rmse05=3.012319e-03, rmse10=5.268434e-03, fit_s=8.7916e-04, factor_s=0.033
ablation windows=(4, 4), r=16, dof=  256, rmse05=2.918712e-03, rmse10=4.371698e-03, fit_s=2.3542e-03, factor_s=0.096
ablation windows=(4, 4), r=32, dof=  512, rmse05=2.800270e-03, rmse10=3.556086e-03, fit_s=3.1979e-03, factor_s=0.319
ablation windows=(4, 4), r=96, dof= 1536, rmse05=2.505735e-03, rmse10=2.594278e-03, fit_s=1.5846e-02, factor_s=2.669
ablation windows=(8, 8), r= 8, dof=  512, rmse05=2.663845e-03, rmse10=3.865322e-03, fit_s=8.7433e-04, factor_s=0.021
ablation windows=(8, 8), r=16, dof= 1024, rmse05=2.521007e-03, rmse10=3.097166e-03, fit_s=2.1767e-03, factor_s=0.087
ablation windows=(8, 8), r=32, dof= 2048, rmse05=2.319755e-03, rmse10=2.472181e-03, fit_s=7.5934e-03, factor_s=0.443
ablation windows=(8, 8), r=96, dof= 6144, rmse05=2.026970e-03, rmse10=1.984794e-03, fit_s=2.8616e-02, factor_s=7.885
ablation windows=(16, 16), r= 8, dof= 2048, rmse05=2.289451e-03,

## Saved PoU checkpoint format

The saved checkpoint `case3_ecmor/hardcurl_pinn_spe10_Q1_64x64_pou.pt` is intentionally **not** a simulator-ready integration.  The simulator will need a matching loader later.

The file contains:

- `format`: string tag `hardcurl_pinn_pou_head_v1`.
- `frozen_state_dict`: the unchanged hidden-layer/final-layer checkpoint state dict copied from `hardcurl_pinn_spe10_Q1_64x64.pt`.
- `pou`: dictionary with the window grid, overlap, smooth cosine-bump type, projection matrix `P` of shape `(96, r)`, selected `r`, fitted `theta` of shape `(K, r)`, ridge parameters, and the nonzero anchor `Theta_bar` where each window falls back to the trained stream-function feature.
- `provenance`: source checkpoint path, source notebook path, simulator path, selected config, and selected metric summary.

Metric tables from Gate 1, Gate 3, and the ablation are also written as JSON next to the checkpoint.


In [ ]:
# Save checkpoint and metric JSON.
# Use the recommended ablation config if RUN_ABLATION produced one; otherwise save the default selected PoU head.
if ablation_results:
    save_window_shape = tuple(recommended["window_shape"])
    save_r = int(recommended["r"])
    save_head = make_pou_head(window_shape=save_window_shape, r=save_r, overlap=POU_OVERLAP)
    # Fit the saved head at t=0 no-drift, matching Gate 1 convention.
    save_fit = save_head.fit(F_cg0, ridge_rel=POU_SELECTED_RIDGE_REL)
    save_theta = save_fit["theta"]
    save_ridge_abs = save_fit["ridge_abs"]
    save_factor_s = save_fit["factor_s"]
else:
    save_window_shape = POU_WINDOW_SHAPE
    save_r = POU_R
    save_head = pou
    save_theta = selected["theta"]
    save_ridge_abs = selected["ridge_abs"]
    save_factor_s = selected["factor_s"]

source_ckpt = torch.load(SOURCE_CHECKPOINT, map_location="cpu", weights_only=False)
checkpoint = {
    "format": "hardcurl_pinn_pou_head_v1",
    "description": "PoU linear head for the frozen hard-curl PINN hidden features; simulator loader not included.",
    "frozen_state_dict": {k: v.detach().cpu() for k, v in source_ckpt["state_dict"].items()},
    "source_architecture": source_ckpt.get("architecture", {}),
    "pou": {
        "window_shape": tuple(save_window_shape),
        "overlap": float(POU_OVERLAP),
        "window_function": "normalized tensor-product C1 cosine bump",
        "P": torch.as_tensor(save_head.P, dtype=torch.float64),
        "r": int(save_r),
        "theta": torch.as_tensor(save_theta, dtype=torch.float64),
        "ridge_lambda_rel": float(POU_SELECTED_RIDGE_REL),
        "ridge_lambda_abs": float(save_ridge_abs),
        "theta_bar": torch.as_tensor(save_head.theta_bar, dtype=torch.float64),
        "factor_s": float(save_factor_s),
    },
    "provenance": {
        "source_checkpoint": str(SOURCE_CHECKPOINT),
        "source_notebook": str(NOTEBOOK_DIR / "LCG_DengGinting_example4_spe10_Q1_no_fracture_hardcurl_PINN.ipynb"),
        "simulator": str(NOTEBOOK_DIR / "impes_spe10_simulator.py"),
        "primary_oracle": ORACLE_NAME,
        "selected_config": {"window_shape": tuple(save_window_shape), "r": int(save_r), "ridge_rel": float(POU_SELECTED_RIDGE_REL)},
    },
}
torch.save(checkpoint, POU_CHECKPOINT)

metrics = {
    "gate0": gate0,
    "gate1": {
        "base_rows": base_rows,
        "ridge_results": [
            {k: v for k, v in r.items() if k not in {"theta", "prediction"}}
            for r in ridge_results
        ],
        "selected_ridge_rel": POU_SELECTED_RIDGE_REL,
        "selected_vs_cg": selected["vs_cg"],
        "selected_vs_oracle": selected["vs_oracle"],
    },
    "gate2": gate2_stats,
    "gate3": gate3_results,
    "ablation": ablation_results,
    "recommended": recommended if ablation_results else None,
    "oracle_diagnostics": oracle_flux_diagnostics,
}
GATE_METRICS_JSON.write_text(json.dumps(json_ready(metrics), indent=2))
print("Saved PoU checkpoint:", POU_CHECKPOINT)
print("Saved metrics JSON:", GATE_METRICS_JSON)
